라이브러리 설치

In [ ]:
!pip install torch transformers tokenizers

KoBERT Model

In [ ]:
# Load model directly
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained("monologg/kobert")
model = AutoModel.from_pretrained("monologg/kobert")

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import numpy as np

dataset class  정의

In [ ]:
class TextClassificationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        label = self.labels[item]

        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

model & tokenizer load

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("monologg/kobert")
model = AutoModelForSequenceClassification.from_pretrained("monologg/kobert", num_labels=3)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at monologg/kobert and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertForSequenceClassification의 일부 가중치는 monologg/kobert의 모델 검사점에서 초기화되지 않았으며 새로 초기화되었습니다. ['classifier.bias', 'classifier.weight']
이 모델을 예측 및 추론에 사용할 수 있도록 다운스트림 작업에 대해 교육해야 합니다.

데이터 준비 (예시입니다.)
 - 우선 성능 확인용으로 진행했습니다.

In [ ]:
# 이 부분은 실제 데이터로 대체해야 합니다
texts = ["관광지 추천해주세요", "버스 시간표 알려주세요", "근처 병원 어디있나요"] * 1000
labels = [0, 1, 2] * 1000  # 0: 관광, 1: 교통, 2: 건강

# 데이터 분할
train_texts, val_texts, train_labels, val_labels = train_test_split(texts, labels, test_size=0.2)

# 데이터셋 생성
train_dataset = TextClassificationDataset(train_texts, train_labels, tokenizer, max_len=128)
val_dataset = TextClassificationDataset(val_texts, val_labels, tokenizer, max_len=128)

# DataLoader 생성
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)

학습 함수 정의

In [ ]:
def train_epoch(model, data_loader, optimizer, device):
    model.train()
    losses = []
    for batch in data_loader:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        losses.append(loss.item())
        loss.backward()
        optimizer.step()
    return np.mean(losses)

def eval_model(model, data_loader, device):
    model.eval()
    predictions = []
    actual_labels = []
    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            outputs = model(input_ids, attention_mask=attention_mask)
            _, preds = torch.max(outputs.logits, dim=1)
            predictions.extend(preds.cpu().tolist())
            actual_labels.extend(labels.cpu().tolist())
    return classification_report(actual_labels, predictions)

학습 실행

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
optimizer = AdamW(model.parameters(), lr=2e-5)

epochs = 3
for epoch in range(epochs):
    train_loss = train_epoch(model, train_loader, optimizer, device)
    print(f'Epoch {epoch+1}/{epochs}, Train Loss: {train_loss:.4f}')
    print(eval_model(model, val_loader, device))

/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Epoch 1/3, Train Loss: 0.2048
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       209
           1       1.00      1.00      1.00       201
           2       1.00      1.00      1.00       190

    accuracy                           1.00       600
   macro avg       1.00      1.00      1.00       600
weighted avg       1.00      1.00      1.00       600

Epoch 2/3, Train Loss: 0.0140
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       209
           1       1.00      1.00      1.00       201
           2       1.00      1.00      1.00       190

    accuracy                           1.00       600
   macro avg       1.00      1.00      1.00       600
weighted avg       1.00      1.00      1.00       600

Epoch 3/3, Train Loss: 0.0057
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       209
           1       1.00      1.00      

모델 저장

In [ ]:
torch.save(model.state_dict(), 'kobert_classifier.pth')

임의의 텍스트 입력 시 분류가 되는지 확인

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 모델과 토크나이저 로드
model_path = "kobert_classifier.pth"  # 저장된 모델 파일 경로
tokenizer = AutoTokenizer.from_pretrained("monologg/kobert")
model = AutoModelForSequenceClassification.from_pretrained("monologg/kobert", num_labels=3)

# 저장된 모델 가중치 로드
model.load_state_dict(torch.load(model_path, map_location=torch.device('cpu')))
model.eval()  # 평가 모드로 설정

# 클래스 레이블 정의 (학습 시 사용한 레이블 순서와 일치해야 함)
class_labels = ['관광', '교통', '건강']

def classify_text(text):
    # 입력 텍스트 토큰화
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128, padding="max_length")

    # 모델 추론
    with torch.no_grad():
        outputs = model(**inputs)

    # 가장 높은 확률의 클래스 선택
    predicted_class = torch.argmax(outputs.logits, dim=1).item()

    # 예측 확률 계산
    probabilities = torch.nn.functional.softmax(outputs.logits, dim=1)
    confidence = probabilities[0][predicted_class].item()

    return class_labels[predicted_class], confidence

# 사용자 입력 받기 및 분류
while True:
    user_input = input("분류할 텍스트를 입력하세요 (종료하려면 'q' 입력): ")
    if user_input.lower() == 'q':
        break

    predicted_class, confidence = classify_text(user_input)
    print(f"예측 클래스: {predicted_class}")
    print(f"신뢰도: {confidence:.2f}")
    print()

print("프로그램을 종료합니다.")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at monologg/kobert and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


분류할 텍스트를 입력하세요 (종료하려면 'q' 입력): 내가 선릉역으로 갈려면 어디로 가야하나?
예측 클래스: 교통
신뢰도: 0.98

분류할 텍스트를 입력하세요 (종료하려면 'q' 입력): 근처 병원이 어디여?
예측 클래스: 관광
신뢰도: 0.93

분류할 텍스트를 입력하세요 (종료하려면 'q' 입력): 약국
예측 클래스: 관광
신뢰도: 1.00

분류할 텍스트를 입력하세요 (종료하려면 'q' 입력): 서울 둘레길 1코스에 대해서 알려줘
예측 클래스: 교통
신뢰도: 0.98

분류할 텍스트를 입력하세요 (종료하려면 'q' 입력): 근처 병원이 어디있나?
예측 클래스: 관광
신뢰도: 0.93

분류할 텍스트를 입력하세요 (종료하려면 'q' 입력): 병원이 어디야?
예측 클래스: 관광
신뢰도: 0.99

분류할 텍스트를 입력하세요 (종료하려면 'q' 입력): q
프로그램을 종료합니다.
